# 02 — Nền UNSW IoT 2016 → bảng gói đã đổi địa chỉ

Theo `PLAN_tron_nen_window.md` mục 4 (bước 02).

- Tách bảng gói 20 ngày UNSW bằng `tshark` (cùng bộ trường với notebook 01).
- Chỉ giữ IPv4 và ARP — giống phạm vi data capture (BPF `ip or arp`).
- Đổi địa chỉ để không đụng lab:
  - MAC gateway UNSW `14:cc:20:51:33:ea` → MAC router lab `de:a6:32:c8:be:ff` (IP `.1` giữ nguyên).
  - IP UNSW trùng `.182` / `.236` → IP LAN còn trống.
- Chia ngày 12/4/4 phân tầng theo ngày nặng/nhẹ và cuối tuần.
- Lập bảng thiết bị (client dùng làm "máy tấn công", camera dùng làm "máy đích") cho bước 03.

Đầu ra: `Data_after_processing/benign_packets/<day>.parquet` (sắp theo thời gian), `reports/02_*.csv`.

In [1]:
import subprocess
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd

ROOT = Path('/home/ubuntu/sepcung/03.Brute_Force_vs_Recon')
UNSW = ROOT / 'Data' / 'public' / 'unsw_iot2016'
OUT = ROOT / 'Data_after_processing'
BEN_DIR = OUT / 'benign_packets'
TMP_DIR = OUT / 'benign_packets_tmp'
REPORT_DIR = OUT / 'reports'
for d in [BEN_DIR, TMP_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

LAB_ROUTER_MAC = 'de:a6:32:c8:be:ff'
UNSW_GW_MAC = '14:cc:20:51:33:ea'
RESERVED_IPS = ['192.168.1.1', '192.168.1.182', '192.168.1.236']
COLLIDE_IPS = ['192.168.1.182', '192.168.1.236']
LAN_PREFIX = '192.168.1.'
N_JOBS = 10
ROW_GROUP = 200_000

SPLIT = {
    'train': ['16-09-23', '16-09-24', '16-09-25', '16-09-27', '16-09-28', '16-09-29',
              '16-09-30', '16-10-03', '16-10-04', '16-10-06', '16-10-08', '16-10-12'],
    'val': ['16-09-26', '16-10-01', '16-10-07', '16-10-10'],
    'test': ['16-10-02', '16-10-05', '16-10-09', '16-10-11'],
}
DAY_SPLIT = {d: s for s, ds in SPLIT.items() for d in ds}

# vai trò thiết bị theo List_Of_Devices.txt của UNSW
CLIENT_MACS = {  # dùng làm "persona" cho máy tấn công
    '74:2f:68:81:69:42': 'Laptop', 'ac:bc:32:d4:6f:2f': 'MacBook', 'b4:ce:f6:a7:a3:c2': 'Android Phone',
    '40:f3:08:ff:1e:da': 'Android Phone 2', 'd0:a6:37:df:a1:e1': 'IPhone', 'f4:5c:89:93:cc:85': 'MacBook/Iphone',
    '08:21:ef:3b:fc:e3': 'Samsung Galaxy Tab',
}
CAMERA_MACS = {  # dùng làm "máy đích" (Pi đóng vai camera)
    '70:ee:50:18:34:43': 'Netatmo Welcome', 'f4:f2:6d:93:51:f1': 'TP-Link Day Night Cloud camera',
    '00:16:6c:ab:6b:88': 'Samsung SmartCam', '30:8c:fb:2f:e4:b2': 'Dropcam', '00:62:6e:51:27:2e': 'Insteon Camera',
    'e8:ab:fa:19:de:4f': 'Insteon Camera (wifi)', '30:8c:fb:b6:ea:45': 'Nest Dropcam',
    '00:24:e4:11:18:a8': 'Withings Smart Baby Monitor',
}
days = sorted(p.stem for p in (UNSW / 'pcap').glob('*.pcap'))
assert sorted(days) == sorted(DAY_SPLIT), 'thiếu ngày UNSW'
len(days)

20

## 1. Tách bảng gói (lượt 1) — cùng bộ trường với notebook 01

In [2]:
FIELDS = ['frame.time_epoch', 'frame.len', 'eth.src', 'eth.dst', 'ip.src', 'ip.dst', 'ip.proto', 'ip.ttl',
          'tcp.srcport', 'tcp.dstport', 'tcp.flags', 'tcp.len', 'udp.srcport', 'udp.dstport', 'icmp.type',
          'arp.opcode', 'arp.src.hw_mac', 'arp.src.proto_ipv4', 'arp.dst.proto_ipv4']
COLS = ['ts', 'len', 'eth_src', 'eth_dst', 'ip_src', 'ip_dst', 'ip_proto', 'ttl',
        'tcp_sport', 'tcp_dport', 'tcp_flags', 'tcp_len', 'udp_sport', 'udp_dport', 'icmp_type',
        'arp_op', 'arp_sha', 'arp_spa', 'arp_tpa']
INT_COLS = ['len', 'ip_proto', 'ttl', 'tcp_sport', 'tcp_dport', 'tcp_len', 'udp_sport', 'udp_dport', 'icmp_type', 'arp_op']


CHUNK_PKTS = 200_000  # tshark giữ trạng thái hội thoại suốt file -> file lớn rất chậm; cắt đoạn (chỉ dùng trường header nên không đổi kết quả)


def read_pcap(pcap):
    chunk_dir = TMP_DIR / f'chunks_{Path(pcap).stem}'
    chunk_dir.mkdir(exist_ok=True)
    subprocess.run(['editcap', '-F', 'pcap', '-c', str(CHUNK_PKTS), str(pcap), str(chunk_dir / 'part.pcap')], check=True)
    lines = []
    for part in sorted(chunk_dir.glob('part*.pcap')):
        cmd = ['tshark', '-r', str(part), '-n', '-T', 'fields', '-E', 'separator=\t', '-E', 'occurrence=f',
               '-o', 'tcp.desegment_tcp_streams:FALSE', '-o', 'ip.defragment:FALSE', '-Y', 'ip or arp']
        for f in FIELDS:
            cmd += ['-e', f]
        res = subprocess.run(cmd, capture_output=True, text=True)
        lines += [l.split('\t') for l in res.stdout.splitlines() if l]
        part.unlink()
    chunk_dir.rmdir()
    df = pd.DataFrame(lines, columns=COLS)
    df['ts'] = pd.to_numeric(df['ts'], errors='coerce')
    for c in INT_COLS:
        df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int32')
    df['tcp_flags'] = df['tcp_flags'].map(lambda x: int(x, 16) if x else pd.NA).astype('Int32')
    is_icmp = df['ip_proto'] == 1
    df.loc[is_icmp, ['tcp_sport', 'tcp_dport', 'tcp_flags', 'tcp_len', 'udp_sport', 'udp_dport']] = pd.NA
    for c in ['eth_src', 'eth_dst', 'arp_sha']:
        df[c] = df[c].str.lower()
    for c in ['ip_src', 'ip_dst', 'arp_spa', 'arp_tpa', 'eth_src', 'eth_dst', 'arp_sha']:
        df[c] = df[c].replace('', pd.NA)
    return df


def extract_day(day):
    df = read_pcap(UNSW / 'pcap' / f'{day}.pcap')
    df.to_parquet(TMP_DIR / f'{day}.parquet', index=False)
    ips = pd.concat([df['ip_src'], df['ip_dst'], df['arp_spa'], df['arp_tpa']]).dropna()
    lan_ips = set(ips[ips.str.startswith(LAN_PREFIX)].unique())
    return day, len(df), lan_ips


with ProcessPoolExecutor(max_workers=N_JOBS) as ex:
    pass1 = list(ex.map(extract_day, days))
all_lan_ips = set().union(*[s for _, _, s in pass1])
print(sum(n for _, n, _ in pass1), 'gói IPv4/ARP;', len(all_lan_ips), 'IP LAN khác nhau')

23097810 gói IPv4/ARP; 37 IP LAN khác nhau


## 2. Bảng đổi IP và MAC

In [3]:
used = all_lan_ips | set(RESERVED_IPS)
free = [f'{LAN_PREFIX}{i}' for i in range(2, 255) if f'{LAN_PREFIX}{i}' not in used]
ip_map = {}
for ip in COLLIDE_IPS:
    if ip in all_lan_ips:
        ip_map[ip] = free.pop(0)
mac_map = {UNSW_GW_MAC: LAB_ROUTER_MAC}
remap = pd.DataFrame([{'kind': 'ip', 'from': k, 'to': v} for k, v in ip_map.items()]
                     + [{'kind': 'mac', 'from': k, 'to': v} for k, v in mac_map.items()])
remap.to_csv(REPORT_DIR / '02_address_remap.csv', index=False)
remap

,kind,from,to
0,ip,192.168.1.236,192.168.1.3
1,mac,14:cc:20:51:33:ea,de:a6:32:c8:be:ff


## 3. Áp dụng đổi địa chỉ, sắp theo thời gian, lưu parquet + bảng thiết bị (lượt 2)

In [4]:
def finalize_day(day):
    df = pd.read_parquet(TMP_DIR / f'{day}.parquet')
    for c in ['ip_src', 'ip_dst', 'arp_spa', 'arp_tpa']:
        df[c] = df[c].replace(ip_map)
    for c in ['eth_src', 'eth_dst', 'arp_sha']:
        df[c] = df[c].replace(mac_map)
    df['label'] = 'benign'
    df = df.sort_values('ts', kind='stable').reset_index(drop=True)
    df.to_parquet(BEN_DIR / f'{day}.parquet', index=False, row_group_size=ROW_GROUP)

    # bảng thiết bị: MAC -> IP chính (theo gói nó gửi), số gói, khoảng hoạt động
    lan_tx = df[df['ip_src'].fillna('').str.startswith(LAN_PREFIX)]
    dev = (lan_tx.groupby('eth_src')
           .agg(ip=('ip_src', lambda s: s.mode().iat[0]), pkts=('ts', 'size'), t0=('ts', 'min'), t1=('ts', 'max'))
           .reset_index().rename(columns={'eth_src': 'mac'}))
    dev['role'] = np.where(dev['mac'].isin(CLIENT_MACS), 'client',
                           np.where(dev['mac'].isin(CAMERA_MACS), 'camera',
                                    np.where(dev['mac'] == LAB_ROUTER_MAC, 'gateway', 'other')))
    dev['device'] = dev['mac'].map({**CLIENT_MACS, **CAMERA_MACS}).fillna('')
    dev['day'] = day
    summ = {'day': day, 'split': DAY_SPLIT[day], 'weekday': pd.Timestamp('20' + day).day_name(),
            'pkts': len(df), 't0': df['ts'].min(), 't1': df['ts'].max(),
            'hours': (df['ts'].max() - df['ts'].min()) / 3600,
            'arp': int(df['arp_op'].notna().sum()), 'tcp': int((df['ip_proto'] == 6).sum()),
            'udp': int((df['ip_proto'] == 17).sum()), 'icmp': int((df['ip_proto'] == 1).sum()),
            'tcp22': int(((df['tcp_sport'] == 22) | (df['tcp_dport'] == 22)).sum()),
            'lan_hosts': dev['mac'].nunique(),
            'clients_active': int((dev['role'] == 'client').sum()),
            'cameras_active': int((dev['role'] == 'camera').sum())}
    (TMP_DIR / f'{day}.parquet').unlink()
    return summ, dev


with ProcessPoolExecutor(max_workers=N_JOBS) as ex:
    pass2 = list(ex.map(finalize_day, days))
TMP_DIR.rmdir()

day_summary = pd.DataFrame([s for s, _ in pass2]).sort_values('day')
devices = pd.concat([d for _, d in pass2], ignore_index=True)
day_summary.to_csv(REPORT_DIR / '02_day_summary.csv', index=False)
devices.to_csv(REPORT_DIR / '02_lan_devices_by_day.csv', index=False)
day_summary

,day,split,weekday,pkts,t0,t1,hours,arp,tcp,udp,icmp,tcp22,lan_hosts,clients_active,cameras_active
0,16-09-23,train,Friday,901108,1.474553e+09,1.474639e+09,23.999369,67134,690787,111794,28381,0,22,4,5
1,16-09-24,train,Saturday,759807,1.474639e+09,1.474726e+09,23.999165,58283,603848,63447,31299,0,18,1,5
2,16-09-25,train,Sunday,498017,1.474726e+09,1.474812e+09,23.999256,55972,347542,60192,31421,0,18,1,5
3,16-09-26,val,Monday,540426,1.474812e+09,1.474898e+09,23.999652,56605,387856,61790,31305,0,19,2,5
4,16-09-27,train,Tuesday,508116,1.474898e+09,1.474985e+09,23.999859,57474,357249,59103,31437,0,18,1,5
5,16-09-28,train,Wednesday,1996955,1.474985e+09,1.475071e+09,23.999650,67611,1784729,108796,32540,608,25,2,7
6,16-09-29,train,Thursday,718432,1.475071e+09,1.475158e+09,23.999437,77255,502003,103729,31947,0,24,3,6
7,16-09-30,train,Friday,783634,1.475158e+09,1.475244e+09,23.999443,75141,563008,110406,31651,0,22,1,6
8,16-10-01,val,Saturday,715734,1.475244e+09,1.475330e+09,23.999339,78530,496609,105565,31605,0,22,1,6
9,16-10-02,test,Sunday,587968,1.475330e+09,1.475413e+09,22.999424,73839,390859,89882,30099,0,22,1,6


In [5]:
split_summary = day_summary.groupby('split').agg(
    days=('day', 'size'), hours=('hours', 'sum'), pkts=('pkts', 'sum'),
    heavy_days=('pkts', lambda s: (s >= 1_300_000).sum()),
    weekend_days=('weekday', lambda s: s.isin(['Saturday', 'Sunday']).sum()))
split_summary.to_csv(REPORT_DIR / '02_split_summary.csv')
split_summary

,days,hours,pkts,heavy_days,weekend_days
split,,,,,
test,4,94.998508,4708541,2,2
train,12,278.473294,15245860,3,3
val,4,95.998584,3143409,1,1


In [6]:
# Kiểm tra: không còn IP .182/.236 gốc của UNSW và không còn MAC gateway UNSW
chk = []
for day in days:
    d = pd.read_parquet(BEN_DIR / f'{day}.parquet', columns=['ip_src', 'ip_dst', 'arp_spa', 'arp_tpa', 'eth_src', 'eth_dst'])
    chk.append({'day': day,
                'ip_182_236_left': int(d[['ip_src', 'ip_dst', 'arp_spa', 'arp_tpa']].isin(COLLIDE_IPS).any(axis=1).sum()),
                'unsw_gw_mac_left': int(d[['eth_src', 'eth_dst']].isin([UNSW_GW_MAC]).any(axis=1).sum())})
chk = pd.DataFrame(chk)
print(chk[['ip_182_236_left', 'unsw_gw_mac_left']].sum().to_dict())
# client/camera hoạt động theo ngày — nguồn persona/máy đích cho bước 03
devices[devices['role'].isin(['client', 'camera'])].pivot_table(index=['role', 'device'], columns='day', values='pkts', aggfunc='sum').fillna(0).astype(int)

{'ip_182_236_left': 0, 'unsw_gw_mac_left': 0}


day                                    16-09-23  16-09-24  16-09-25  16-09-26  \
role   device                                                                   
camera Dropcam                           118868     80025     81480     95785   
       Insteon Camera                         0         0         0         0   
       Insteon Camera (wifi)                  0         0         0         0   
       Nest Dropcam                           0         0         0         0   
       Netatmo Welcome                    16142      7190      6768     15629   
       Samsung SmartCam                   43434     36536     36449     40001   
       TP-Link Day Night Cloud camera      4752      5274      4652      4872   
       Withings Smart Baby Monitor        19660     19076     19163     19527   
client Android Phone                       8693         0         0         0   
       Android Phone 2                        0         0         0         0   
       IPhone                               311         0         0       381   
       Laptop                             89241         0         0         0   
       MacBook                                0         0         0         0   
       MacBook/Iphone                         0         0         0         0   
       Samsung Galaxy Tab                 14033      3315      2818      3908   

day                                    16-09-27  16-09-28  16-09-29  16-09-30  \
role   device                                                                   
camera Dropcam                           102263    115247     96757    108826   
       Insteon Camera                         0     90072     26930     27242   
       Insteon Camera (wifi)                  0       318         0         0   
       Nest Dropcam                           0         0         0         0   
       Netatmo Welcome                    11239     38149     25171     13272   
       Samsung SmartCam                   40072     46689     39832     38283   
       TP-Link Day Night Cloud camera      4664     45537      5914     68529   
       Withings Smart Baby Monitor        19733     20041     19490     19854   
client Android Phone                          0         0         0         0   
       Android Phone 2                        0         0      2513         0   
       IPhone                                 0         0         0         0   
       Laptop                                 0         0         0         0   
       MacBook                                0    405081     13116         0   
       MacBook/Iphone                         0         0         0         0   
       Samsung Galaxy Tab                  4121     69008      7196      2282   

day                                    16-10-01  16-10-02  16-10-03  16-10-04  \
role   device                                                                   
camera Dropcam                           113952     78177     98045    114410   
       Insteon Camera                     30504     25092     26116     25887   
       Insteon Camera (wifi)                  0         0         0         0   
       Nest Dropcam                           0         0         0         0   
       Netatmo Welcome                    20143      7239     10025     19819   
       Samsung SmartCam                   38777     34028     35642     39761   
       TP-Link Day Night Cloud camera      5926      4763      4680      5301   
       Withings Smart Baby Monitor        20127     18993     20006     19976   
client Android Phone                          0         0         0       241   
       Android Phone 2                        0         0         0         0   
       IPhone                                 0         0         0       239   
       Laptop                                 0         0         0         0   
       MacBook                                0         0         0    279184   
       MacBook/Iphone        